<a href="https://colab.research.google.com/github/kumarchiranj/Robo-speaker/blob/main/advanced_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. आवश्यक लाइब्रेरीज़ इंस्टॉल करें
!pip install fastapi uvicorn nest-asyncio pyngrok faiss-cpu sentence-transformers langchain-text-splitters groq


In [2]:
import os
import json
import asyncio
import nest_asyncio
import numpy as np
import faiss
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from groq import Groq

# Colab में FastAPI रन करने के लिए लूप को पैच करें
nest_asyncio.apply()

# 1. Groq API Key और कॉन्फिगरेशन सेट करें
# (अपनी Groq API Key यहाँ डालें या Colab Secrets में सेव करें)
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
client = Groq()

app = FastAPI(title="Advanced RAG Production System")

# Add a simple root endpoint for health check
@app.get("/")
async def root():
    return {"message": "Welcome to the Advanced RAG Production System API! Visit /docs for API documentation."}

# 2. एम्बेडिंग मॉडल लोड करें (Local CPU पर तेज़ परफॉरमेंस के लिए)
print("Loading Embedding Model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
VECTOR_DIM = 384 # MiniLM की डायमेंशन

# 3. इन-मेमोरी डेटा स्टोर और FAISS इंडेक्स इनिशियलाइज़ करें
faiss_index = faiss.IndexFlatL2(VECTOR_DIM)
metadata_store = {} # chunk_id -> {text, source_doc}
chunk_counter = 0

# 4. API के लिए Pydantic मॉडल्स
class IngestRequest(BaseModel):
    document_text: str
    doc_name: str

class QueryRequest(BaseModel):
    question: str

# ----------------------------------------------------
# ENDPOINT 1: Advanced Document Ingestion (Chunking + FAISS Indexing)
# ----------------------------------------------------
@app.post("/ingest")
async def ingest_document(request: IngestRequest):
    global chunk_counter, faiss_index, metadata_store

    try:
        # Advanced Chunking: ओवरलैप के साथ टेक्स्ट को छोटे हिस्सों में तोड़ना
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )
        chunks = text_splitter.split_text(request.document_text)

        if not chunks:
            raise HTTPException(status_code=400, detail="Document text is empty or too short.")

        # टेक्स्ट चंक्स के लिए एम्बेडिंग्स जनरेट करें
        embeddings = embedding_model.encode(chunks)
        embeddings_np = np.array(embeddings).astype('float32')

        # FAISS इंडेक्स में वेक्टर्स जोड़ें
        faiss_index.add(embeddings_np)

        # मेटाडेटा स्टोर में चंक्स को मैप करें
        for i, chunk in enumerate(chunks):
            metadata_store[chunk_counter] = {
                "text": chunk,
                "source": request.doc_name
            }
            chunk_counter += 1

        return {"status": "Success", "chunks_indexed": len(chunks), "total_vectors": faiss_index.ntotal}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ----------------------------------------------------
# ENDPOINT 2: Advanced Retrieval & Generation (RAG Pipeline)
# ----------------------------------------------------
@app.post("/query")
async def query_rag(request: QueryRequest):
    global faiss_index, metadata_store

    if faiss_index.ntotal == 0:
        raise HTTPException(status_code=400, detail="No documents found in Vector DB. Please ingest data first.")

    try:
        # Step 1: सवाल को वेक्टर में बदलें (Semantic Search)
        query_vector = embedding_model.encode([request.question])
        query_vector_np = np.array(query_vector).astype('float32')

        # Step 2: Top-3 सबसे प्रासंगिक चंक्स रिट्रीव करें
        k = 3
        distances, indices = faiss_index.search(query_vector_np, k)

        retrieved_contexts = []
        sources = []

        for idx in indices[0]:
            if idx in metadata_store:
                retrieved_contexts.append(metadata_store[idx]["text"])
                sources.append(metadata_store[idx]["source"])

        context_str = "\n---\n".join(retrieved_contexts)

        # Step 3: Prompt Engineering - सिस्टम प्रॉम्प्ट और कॉन्टेक्स्ट असेंबली
        system_prompt = (
            "You are an advanced enterprise AI Assistant. Answer the user's question "
            "STRICTLY using the provided context. If the answer cannot be found in the context, "
            "say 'I don't have this information in my knowledge base.' Do not hallucinate."
        )

        user_prompt = f"Context:\n{context_str}\n\nQuestion: {request.question}\nAnswer:"

        # Step 4: Groq API के ज़रिये Llama 3 मॉडल से रिस्पांस लें
        completion = client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.2 # कम क्रिएटिविटी ताकि फैक्ट्स सटीक रहें
        )

        llm_response = completion.choices[0].message.content

        return {
            "answer": llm_response,
            "sources": list(set(sources)),
            "retrieved_chunks": retrieved_contexts
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Loading Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from pyngrok import ngrok
import uvicorn
import asyncio
import threading
import time
import sys # Needed for SystemExit handling

# Declare global variables for the server and tunnel
global server_thread
server_thread = None
global public_url_ngrok
public_url_ngrok = None

# Ensure the 'app' object is accessible from the main notebook scope
# 'app' is defined in cell YaCihEE06NqB, so it should be available.

# 1. Kill any existing ngrok tunnels first to free up the port
print("Attempting to kill existing ngrok tunnels...")
ngrok.kill()
print("Existing ngrok tunnels killed (if any).")

# 1. अपना Ngrok Authtoken सेट करें
NGROK_AUTH_TOKEN = "3Fc1rIF1EjdSQzTCfGUlDKSQ07l_8afd6s48V1mMBVSMcozwB"
if NGROK_AUTH_TOKEN == "YOUR_NGROQ_API_KEY_HERE": # Check if the placeholder is still there
    print("Warning: NGROK_AUTH_TOKEN is not set. Please replace 'YOUR_NGROK_AUTH_TOKEN_HERE' with your actual ngrok authtoken in cell `kgIcWJXK6jSq`.")
    # Optionally, raise an error or prevent further execution if authtoken is crucial
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


# 3. FastAPI सर्वर को बैकग्राउंड में चालू करें using a separate thread
# Only start the server thread if it's not already running
if server_thread is None or not server_thread.is_alive():
    print("Starting Uvicorn server in background thread...")
    def run_uvicorn():
        # Create a new event loop for this thread and apply nest_asyncio to it
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        import nest_asyncio # Import here to ensure it applies to this thread's loop
        nest_asyncio.apply(loop)

        config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
        server = uvicorn.Server(config)
        try:
            loop.run_until_complete(server.serve())
        except SystemExit as e:
            if e.code == 1: # Common code for uvicorn failing to bind
                print(f"Uvicorn server thread: SystemExit(1) caught during startup (e.g., port in use). Thread exiting gracefully.")
            else:
                print(f"Uvicorn server thread: Caught SystemExit with code {e.code}. Re-raising.")
                raise # Re-raise other SystemExit codes
        except OSError as e:
            if e.errno == 98: # Errno 98 is 'Address already in use'
                print(f"Uvicorn server thread: Port 8000 is already in use. Server could not start. You may need to restart the Colab runtime.")
            else:
                print(f"Uvicorn server thread: OSError encountered: {e}")
        except Exception as e:
            print(f"Uvicorn server thread: An unexpected error occurred: {e}")
        finally:
            print("Uvicorn background thread finished.")

    server_thread = threading.Thread(target=run_uvicorn)
    server_thread.daemon = True # Allow program to exit even if thread is running
    server_thread.start()

    # Give the server a moment to start up
    time.sleep(5) # Adjust as needed
    print("Uvicorn server started in a background thread. Check the output above for any startup errors.")
else:
    print("Uvicorn server thread is already running.")

# 2. टनल खोलें (Port 8000 पर) - This should happen AFTER Uvicorn is attempting to listen
if public_url_ngrok is None or not public_url_ngrok.is_connected():
    try:
        # Ensure the server is ready before trying to connect ngrok
        if server_thread and server_thread.is_alive():
            public_url_ngrok = ngrok.connect(8000)
            print(f"\n🚀 Your Advanced RAG Pipeline is LIVE at:")
            print(f"👉 Public URL: {public_url_ngrok}")
            print(f"👉 Interactive Docs (Swagger UI): {public_url_ngrok}/docs\n")
        else:
            print("Ngrok cannot connect: Uvicorn server did not start successfully.")
            public_url_ngrok = None
    except Exception as e:
        print(f"Error establishing ngrok tunnel: {e}")
        public_url_ngrok = None # Reset if tunnel failed
        print("Please check your ngrok authtoken and ensure ngrok is installed correctly.")
else:
    print(f"\n🚀 Server already running. Access at:")
    print(f"👉 Public URL: {public_url_ngrok}")
    print(f"👉 Interactive Docs (Swagger UI): {public_url_ngrok}/docs\n")

# Make public_url_ngrok available as 'public_url' for the next cell that uses it
public_url = public_url_ngrok


INFO:     Started server process [16828]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Attempting to kill existing ngrok tunnels...
Existing ngrok tunnels killed (if any).
Starting Uvicorn server in background thread...
Uvicorn server started in a background thread. Check the output above for any startup errors.

🚀 Your Advanced RAG Pipeline is LIVE at:
👉 Public URL: NgrokTunnel: "https://trunks-tightrope-edginess.ngrok-free.dev" -> "http://localhost:8000"
👉 Interactive Docs (Swagger UI): NgrokTunnel: "https://trunks-tightrope-edginess.ngrok-free.dev" -> "http://localhost:8000"/docs



In [4]:
import requests
import json

# Get the public URL from ngrok output
public_url_object = public_url # 'public_url' variable is available from previous cell
base_url = public_url_object.public_url

# Define the ingest endpoint URL
ingest_url = f"{base_url}/ingest"

# Test the root endpoint first for a health check
print(f"Testing root endpoint: {base_url}/")
root_response = requests.get(base_url)
print("Root API Response Status Code:", root_response.status_code)
try:
    print("Root API Response Body:", root_response.json())
except json.JSONDecodeError:
    print("Root API Response Body (not JSON):", root_response.text)

print("\nNow testing /ingest endpoint...")

# Sample document data
sample_document = {
    "document_text": "LangChain is a framework designed to simplify the creation of applications using large language models. It provides abstractions for common LLM tasks and chains, which are sequences of calls to LLMs or other utilities. LangChain enables building various applications, such as chatbots, summarization tools, and more.",
    "doc_name": "LangChain_Overview"
}

# Send the POST request to /ingest
headers = {'Content-Type': 'application/json'}
response = requests.post(ingest_url, headers=headers, data=json.dumps(sample_document))

# Print the response for /ingest
print("Ingest API Response Status Code:", response.status_code)
try:
    print("Ingest API Response Body:", response.json())
except json.JSONDecodeError:
    print("Ingest API Response Body (not JSON):", response.text)


Testing root endpoint: https://trunks-tightrope-edginess.ngrok-free.dev/
INFO:     34.182.212.197:0 - "GET / HTTP/1.1" 200 OK
Root API Response Status Code: 200
Root API Response Body: {'message': 'Welcome to the Advanced RAG Production System API! Visit /docs for API documentation.'}

Now testing /ingest endpoint...
INFO:     34.182.212.197:0 - "POST /ingest HTTP/1.1" 200 OK
Ingest API Response Status Code: 200
Ingest API Response Body: {'status': 'Success', 'chunks_indexed': 1, 'total_vectors': 1}


### Add another document to the FAISS index

In [5]:
import requests
import json

# New sample document data
new_sample_document = {
    "document_text": "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that would not fit in RAM. It also provides supporting code for evaluation and parameter tuning. FAISS is written in C++ with complete wrappers for Python. Some of the most useful algorithms for FAISS include: IndexFlat (exact search), IndexLSH (locality-sensitive hashing), and IndexIVF (inverted file index).",
    "doc_name": "FAISS_Overview"
}

print("Now ingesting a new document for FAISS_Overview...")

# Send the POST request to /ingest with the new document
response_new = requests.post(ingest_url, headers=headers, data=json.dumps(new_sample_document))

# Print the response for the new /ingest
print("New Ingest API Response Status Code:", response_new.status_code)
try:
    print("New Ingest API Response Body:", response_new.json())
except json.JSONDecodeError:
    print("New Ingest API Response Body (not JSON):", response_new.text)

Now ingesting a new document for FAISS_Overview...
INFO:     34.182.212.197:0 - "POST /ingest HTTP/1.1" 200 OK
New Ingest API Response Status Code: 200
New Ingest API Response Body: {'status': 'Success', 'chunks_indexed': 2, 'total_vectors': 3}


### Query the RAG Pipeline

अब जब हमने कुछ दस्तावेज़ इंडेक्स कर लिए हैं, तो चलिए RAG पाइपलाइन को क्वेरी करके देखें।

In [6]:
import requests
import json

query_url = f"{base_url}/query"

# Sample query data
query_data = {
    "question": "What is LangChain and what can it be used for?"
}

print(f"Querying the RAG pipeline with question: \"{query_data['question']}\"")

# Send the POST request to /query
query_response = requests.post(query_url, headers=headers, data=json.dumps(query_data))

# Print the response
print("Query API Response Status Code:", query_response.status_code)
try:
    print("Query API Response Body:", query_response.json())
except json.JSONDecodeError:
    print("Query API Response Body (not JSON):", query_response.text)

Querying the RAG pipeline with question: "What is LangChain and what can it be used for?"
INFO:     34.182.212.197:0 - "POST /query HTTP/1.1" 500 Internal Server Error
Query API Response Status Code: 500
Query API Response Body: {'detail': "Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}"}
